# Revela Network: treino no Colab

Treina o modelo em 3 etapas com o MIT-Adobe FiveK, usando os 5 especialistas como 5 estilos:

1. **Cena**: backbone, cabeças (iluminante, exposição, material, ruído) e reconstrução de altas luzes.
2. **Base**: renderizador aprendendo o estilo do especialista C.
3. **LoRAs**: uma LoRA para cada um dos outros especialistas (A, B, D, E).

Use **Ambiente de execução → Alterar tipo → GPU (T4)**. Os dados e os pesos ficam no seu Google Drive, então dá para parar e retomar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = "Dimitri-Code56/revela_network"   # troque se o repo tiver outro nome
!git clone -q https://github.com/{REPO}.git projeto 2>/dev/null || git -C projeto pull -q
%cd projeto
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/revela_network'
DADOS_TREINO, DADOS_VAL, PESOS = f'{BASE}/fivek/treino', f'{BASE}/fivek/validacao', f'{BASE}/pesos'

## 1. Baixar e pré-processar o FiveK

Cada foto baixa o DNG e os 5 TIFFs dos especialistas (mais de 100 MB no total), reduz para 512 px e salva um `.npz` de poucos MB no Drive. Os originais são apagados em seguida.

`N_TREINO = 500` costuma levar de 1 a 3 horas, dependendo da velocidade do servidor do MIT. Se a sessão cair, rode a célula de novo: ela continua de onde parou.

In [ ]:
from renderizador.dados import preparar_fivek
N_TREINO, N_VAL = 500, 60
preparar_fivek(DADOS_VAL, 'validacao', n=N_VAL, workers=16)
preparar_fivek(DADOS_TREINO, 'treino', n=N_TREINO, workers=16)

## 2. Treinar

`--iters` vale para as etapas de cena e base; cada LoRA usa um quarto disso. Com 15 000 iterações, a T4 leva algumas horas no total. Para um teste rápido, use `--iters 2000`.

In [ ]:
!python -m renderizador.treino --dados {DADOS_TREINO} --validacao {DADOS_VAL} --etapa tudo \
    --iters 15000 --batch 8 --recorte 384 --workers 2 --log 250 --saida {PESOS}

### Retomar se a sessão cair

- Caiu durante a etapa **base**: `--etapa base --retomar {PESOS}/cena.pt`
- Caiu durante as **LoRAs**: `--etapa lora --retomar {PESOS}/base.pt`

In [ ]:
# !python -m renderizador.treino --dados {DADOS_TREINO} --validacao {DADOS_VAL} --etapa lora --retomar {PESOS}/base.pt --iters 15000 --saida {PESOS}

## 3. Resultados

In [ ]:
import json
m = json.load(open(f'{PESOS}/metricas.json'))
print(f"{'modelo -> alvo':18s} {'PSNR':>8s} {'ΔE':>6s}")
for k, v in m.items():
    print(f"{k:18s} {v['psnr']:8.2f} {v['delta_e']:6.2f}")

Compare `sem_lora->a` com `a->a`: a diferença é o quanto a LoRA de 8 mil parâmetros aproximou o estilo do especialista A.

## 4. Aplicar numa foto sua

Envie um DNG (melhor) ou um JPG. A grade mostra a foto sem LoRA, cada estilo e uma mistura.

In [ ]:
from google.colab import files
from IPython.display import Image as Img, display
foto = list(files.upload())[0]
!python -m renderizador.usar "{foto}" --pesos {PESOS} --grade grade.jpg --estilos a=0.5,e=0.5 --saida mistura.jpg
display(Img('grade.jpg'))